# Problem: route text to the right team automatically.
# Success: beat a dumb baseline by a lot; aim for 85%+ accuracy.
# Wrong prediction cost: low (a human can re-route).

In [1]:
from sklearn.datasets import fetch_20newsgroups
from sklearn.model_selection import train_test_split

In [2]:
cats = ["sci.med", "comp.graphics", "rec.autos", "talk.politics.guns"]
data = fetch_20newsgroups(subset="all", categories=cats,
                          remove=("headers", "footers", "quotes"))

In [3]:
print(len(data.data), "documents")
print(data.target_names)
print(data.data[0][:300])

3863 documents
['comp.graphics', 'rec.autos', 'sci.med', 'talk.politics.guns']
Looking for a graphics/CAD/or-whatever package on a X-Unix box that will
take a file with records like:

n  a  b  p

where n = a count  - integer 
      a = entity a - string
      b = entity b - string
      p = type     - string

and produce a networked graph with nodes represented with boxes or c


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42, stratify=data.target)

In [5]:
from sklearn.dummy import DummyClassifier
base = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print("Baseline accuracy:", base.score(X_test, y_test))

Baseline accuracy: 0.25614489003880986


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

model = Pipeline([
    ("tfidf", TfidfVectorizer(stop_words="english")),
    ("clf", LogisticRegression(max_iter=1000)),
])
model.fit(X_train, y_train)
print("Model accuracy:", model.score(X_test, y_test))

Model accuracy: 0.9172056921086675


In [7]:
from sklearn.metrics import classification_report, confusion_matrix
preds = model.predict(X_test)
print(classification_report(y_test, preds, target_names=data.target_names))
print(confusion_matrix(y_test, preds))

# Error analysis: read what it got wrong
wrong = [i for i in range(len(preds)) if preds[i] != y_test[i]][:5]
for i in wrong:
    print("TRUE:", data.target_names[y_test[i]], "| PRED:", data.target_names[preds[i]])
    print(X_test[i][:200].replace("\n", " "), "\n")

                    precision    recall  f1-score   support

     comp.graphics       0.95      0.93      0.94       195
         rec.autos       0.87      0.93      0.90       198
           sci.med       0.91      0.91      0.91       198
talk.politics.guns       0.94      0.90      0.92       182

          accuracy                           0.92       773
         macro avg       0.92      0.92      0.92       773
      weighted avg       0.92      0.92      0.92       773

[[181   6   6   2]
 [  5 185   5   3]
 [  5   8 180   5]
 [  0  13   6 163]]
TRUE: talk.politics.guns | PRED: rec.autos
 

TRUE: rec.autos | PRED: comp.graphics
Does anyone have a listing of Pontiac's three-letter option codes and what they mean?  Thanks.  -Oliver --- ---------------------------------------------------------------------------- Oliver Scholz   

TRUE: sci.med | PRED: talk.politics.guns
Harvested to order?  

TRUE: sci.med | PRED: rec.autos
Then there are always osteopathy colleges....  

TRUE: sc

In [8]:
import joblib
joblib.dump(model, "router.joblib")

loaded = joblib.load("router.joblib")
text = ["My car makes a strange noise when I brake"]
print(data.target_names[loaded.predict(text)[0]])

rec.autos
